# Βήμα 3 – Σύστημα Α: νευρωνικό δίκτυο πολλαπλών εξόδων (UNSW-NB15)
Ένα δίκτυο, τρεις έξοδοι:
1. **label** – επίθεση ή κανονική ροή → δίνει την πιθανότητα P(attack)
2. **attack_cat** – τύπος (Normal + 9 τύποι επίθεσης)
3. **severity** – σοβαρότητα (None / Low / Medium / High)

και από αυτές υπολογίζεται το **Risk Score = 100 · P(attack) · Severity Weight**.

Προϋπόθεση: να έχει τρέξει το `03_preprocess_unsw.ipynb`.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import keras
from keras import layers
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)

DATA, MODELS, FIG = "../data/", "../models/", "../figures/"
SEED = 42
keras.utils.set_random_seed(SEED)      # αναπαραγώγιμα αποτελέσματα
print("Keras", keras.__version__)

## 3.1 Φόρτωση των έτοιμων δεδομένων

In [ ]:
d = np.load(DATA + "unsw_prepared.npz")
with open(MODELS + "unsw_meta.json", encoding="utf-8") as f:
    meta = json.load(f)

ATTACK_CLASSES = meta["attack_classes"]
SEVERITY_CLASSES = meta["severity_classes"]
SEV_WEIGHTS = np.array([meta["severity_weight"][c] for c in SEVERITY_CLASSES], dtype="float32")

X_train, X_val, X_test = d["X_train"], d["X_val"], d["X_test"]

def targets(split):
    # λίστα με τους 3 στόχους, με την ίδια σειρά που έχουν οι έξοδοι του δικτύου
    return [d[f"y_{split}_label"].astype("float32").reshape(-1, 1),
            d[f"y_{split}_attack_cat"].astype("int32"),
            d[f"y_{split}_severity"].astype("int32")]

y_train, y_val, y_test = targets("train"), targets("val"), targets("test")
n_features = X_train.shape[1]
print("train:", X_train.shape, "| val:", X_val.shape, "| test:", X_test.shape)

## 3.2 Βάρη κλάσεων
Οι τύποι επίθεσης είναι πολύ άνισοι (π.χ. Worms ~100 δείγματα, Normal ~44.000). Χωρίς βάρη το δίκτυο θα αγνοούσε τους σπάνιους τύπους.
Δίνουμε σε κάθε κλάση βάρος ∝ 1/√(συχνότητα): οι σπάνιες κλάσεις «μετράνε» περισσότερο στο σφάλμα, χωρίς υπερβολές.

In [ ]:
def sqrt_balanced_weights(y, n_classes):
    counts = np.bincount(y, minlength=n_classes).astype("float64")
    w = np.sqrt(len(y) / (n_classes * np.maximum(counts, 1)))
    return (w / np.average(w, weights=counts)).astype("float32")   # μέσο βάρος = 1

w_attack = sqrt_balanced_weights(y_train[1], len(ATTACK_CLASSES))
w_sev = sqrt_balanced_weights(y_train[2], len(SEVERITY_CLASSES))
print(pd.Series(w_attack, index=ATTACK_CLASSES).round(2).to_string())
print(pd.Series(w_sev, index=SEVERITY_CLASSES).round(2).to_string())

In [ ]:
def weighted_sparse_ce(class_weights):
    """Sparse categorical cross-entropy όπου κάθε δείγμα ζυγίζεται με το βάρος της κλάσης του."""
    w = keras.ops.convert_to_tensor(class_weights, dtype="float32")
    def loss(y_true, y_pred):
        y = keras.ops.cast(keras.ops.reshape(y_true, (-1,)), "int32")
        ce = keras.losses.sparse_categorical_crossentropy(y, y_pred)
        return ce * keras.ops.take(w, y)
    return loss

## 3.3 Αρχιτεκτονική
Κοινός «κορμός» (3 κρυφά επίπεδα) που μαθαίνει μια αναπαράσταση της ροής, και πάνω του 3 «κεφαλές», μία για κάθε έξοδο.
- **BatchNormalization**: σταθεροποιεί και επιταχύνει την εκπαίδευση.
- **Dropout**: «σβήνει» τυχαία νευρώνες στην εκπαίδευση ώστε το δίκτυο να μην αποστηθίζει (overfitting).

In [ ]:
def build_model(n_features, units=(128, 64, 32), dropout=0.3, lr=1e-3):
    inp = keras.Input(shape=(n_features,), name="flow_features")
    x = inp
    for i, u in enumerate(units):
        x = layers.Dense(u, activation="relu", name=f"dense_{i+1}")(x)
        x = layers.BatchNormalization(name=f"bn_{i+1}")(x)
        x = layers.Dropout(dropout, name=f"dropout_{i+1}")(x)

    out_label = layers.Dense(1, activation="sigmoid", name="label")(x)
    out_attack = layers.Dense(len(ATTACK_CLASSES), activation="softmax", name="attack_cat")(x)
    out_sev = layers.Dense(len(SEVERITY_CLASSES), activation="softmax", name="severity")(x)

    model = keras.Model(inp, [out_label, out_attack, out_sev], name="unsw_multitask")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss=["binary_crossentropy", weighted_sparse_ce(w_attack), weighted_sparse_ce(w_sev)],
        loss_weights=[1.0, 1.0, 0.5],
        metrics=[[keras.metrics.BinaryAccuracy(name="acc"), keras.metrics.AUC(name="auc")],
                 [keras.metrics.SparseCategoricalAccuracy(name="acc")],
                 [keras.metrics.SparseCategoricalAccuracy(name="acc")]],
    )
    return model

model = build_model(n_features)
model.summary()

## 3.4 Εκπαίδευση
- **EarlyStopping**: σταματά όταν το σφάλμα στο validation δεν βελτιώνεται για 8 εποχές και κρατά τα καλύτερα βάρη.
- **ReduceLROnPlateau**: μισό learning rate όταν «κολλάει» για 3 εποχές.

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5),
]
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=60, batch_size=256,
    callbacks=callbacks, verbose=2,
)
hist = pd.DataFrame(history.history)
hist.to_csv(MODELS + "unsw_multitask_history.csv", index=False)
model.save(MODELS + "unsw_multitask.keras")
print("Εποχές που έτρεξαν:", len(hist), "| καλύτερη εποχή:", int(hist["val_loss"].idxmin()) + 1)

## 3.5 Καμπύλες εκπαίδευσης
Αν η γραμμή validation ακολουθεί τη γραμμή train, το δίκτυο γενικεύει. Αν το train συνεχίζει να βελτιώνεται ενώ το validation χειροτερεύει → overfitting.

In [ ]:
acc_cols = [c for c in hist.columns if c.endswith("_acc") and not c.startswith("val_")]
panels = ["loss"] + acc_cols
fig, axes = plt.subplots(1, len(panels), figsize=(4.6 * len(panels), 3.8))
for ax, c in zip(np.atleast_1d(axes), panels):
    ax.plot(hist.index + 1, hist[c], color="#2a78d6", lw=2, label="train")
    ax.plot(hist.index + 1, hist["val_" + c], color="#eb6834", lw=2, label="validation")
    ax.set_title(c); ax.set_xlabel("Εποχή"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(FIG + "14_unsw_training_curves.png", dpi=200); plt.show()

## 3.6 Προβλέψεις στο test set (δεδομένα που το δίκτυο δεν έχει δει ποτέ)

In [ ]:
p_label, p_attack, p_sev = model.predict(X_test, batch_size=1024, verbose=0)
p_label = p_label.ravel()                      # P(attack) για κάθε ροή

pred_label = (p_label >= 0.5).astype(int)
pred_attack = p_attack.argmax(1)
pred_sev = p_sev.argmax(1)
true_label, true_attack, true_sev = y_test[0].ravel().astype(int), y_test[1], y_test[2]

## 3.7 Έξοδος 1 – Ανίχνευση επίθεσης

In [ ]:
res_label = {
    "accuracy": accuracy_score(true_label, pred_label),
    "precision": precision_score(true_label, pred_label),
    "recall": recall_score(true_label, pred_label),
    "f1": f1_score(true_label, pred_label),
    "roc_auc": roc_auc_score(true_label, p_label),
}
print(pd.Series(res_label).round(4).to_string())

def plot_cm(y_true, y_pred, names, title, fname, size=(5, 4.2)):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(names)))
    cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
    annot = np.array([[f"{cmn[i, j]:.0%}\n{cm[i, j]}" for j in range(len(names))] for i in range(len(names))])
    fig, ax = plt.subplots(figsize=size)
    sns.heatmap(cmn, annot=annot, fmt="", cmap="Blues", vmin=0, vmax=1, cbar=False,
                xticklabels=names, yticklabels=names, annot_kws={"size": 8}, ax=ax)
    ax.set_xlabel("Πρόβλεψη"); ax.set_ylabel("Πραγματικό"); ax.set_title(title)
    plt.xticks(rotation=45, ha="right"); plt.yticks(rotation=0)
    plt.tight_layout(); plt.savefig(FIG + fname, dpi=200); plt.show()

plot_cm(true_label, pred_label, ["Normal", "Attack"], "Ανίχνευση επίθεσης (test)", "15_unsw_cm_label.png")

## 3.8 Έξοδος 2 – Τύπος επίθεσης
`macro avg` = μέσος όρος όλων των κλάσεων με ίδιο βάρος (δείχνει πόσο καλά πάμε και στις σπάνιες)· `weighted avg` = ζυγισμένος με το πλήθος.

In [ ]:
print(classification_report(true_attack, pred_attack, labels=range(len(ATTACK_CLASSES)),
                            target_names=ATTACK_CLASSES, digits=3, zero_division=0))
plot_cm(true_attack, pred_attack, ATTACK_CLASSES, "Τύπος επίθεσης (test)", "16_unsw_cm_attack_cat.png", size=(9.5, 8))

## 3.9 Έξοδος 3 – Σοβαρότητα

In [ ]:
print(classification_report(true_sev, pred_sev, labels=range(len(SEVERITY_CLASSES)),
                            target_names=SEVERITY_CLASSES, digits=3, zero_division=0))
plot_cm(true_sev, pred_sev, SEVERITY_CLASSES, "Σοβαρότητα (test)", "17_unsw_cm_severity.png", size=(5.5, 4.6))

## 3.10 Risk Score
**Risk = 100 · P(attack) · Severity Weight**

Το Severity Weight είναι το αναμενόμενο βάρος σοβαρότητας *δεδομένου ότι πρόκειται για επίθεση*: παίρνουμε τις πιθανότητες Low/Medium/High της 3ης εξόδου,
τις κανονικοποιούμε ώστε να αθροίζουν σε 1 και υπολογίζουμε 0.33·P(Low) + 0.66·P(Medium) + 1.0·P(High).

In [ ]:
def risk_score(p_label, p_sev):
    p_att_sev = p_sev[:, 1:]                                        # Low, Medium, High (χωρίς το None)
    sev_w = (p_att_sev * SEV_WEIGHTS[1:]).sum(1) / np.clip(p_att_sev.sum(1), 1e-9, None)
    return (100 * p_label * sev_w).astype("float64")

risk = risk_score(p_label, p_sev)
risk_df = pd.DataFrame({"attack_cat": np.array(ATTACK_CLASSES)[true_attack],
                        "severity": np.array(SEVERITY_CLASSES)[true_sev], "risk": risk})
order = ["None", "Low", "Medium", "High"]
print(risk_df.groupby("severity")["risk"].describe().loc[order].round(1)[["count", "mean", "25%", "50%", "75%"]])

fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=risk_df, x="severity", y="risk", order=order, color="#86b6ef", showfliers=False, width=0.5, ax=ax)
ax.set_xlabel("Πραγματική σοβαρότητα"); ax.set_ylabel("Risk Score (0–100)"); ax.set_title("Risk Score ανά πραγματική σοβαρότητα (test)")
plt.tight_layout(); plt.savefig(FIG + "18_unsw_risk_score.png", dpi=200); plt.show()

## 3.11 Παράδειγμα: τι βγάζει το σύστημα για μεμονωμένες ροές

In [ ]:
rng = np.random.default_rng(SEED)
idx = rng.choice(len(X_test), 10, replace=False)
pd.DataFrame({
    "Πραγματικό": np.array(ATTACK_CLASSES)[true_attack[idx]],
    "P(attack)": p_label[idx].round(3),
    "Πρόβλεψη": np.where(pred_label[idx] == 1, "Attack", "Normal"),
    "Τύπος": np.array(ATTACK_CLASSES)[pred_attack[idx]],
    "Σοβαρότητα": np.array(SEVERITY_CLASSES)[pred_sev[idx]],
    "Risk Score": risk[idx].round(1),
})

In [ ]:
results = {
    "label": res_label,
    "attack_cat": {"accuracy": accuracy_score(true_attack, pred_attack),
                   "macro_f1": f1_score(true_attack, pred_attack, average="macro"),
                   "weighted_f1": f1_score(true_attack, pred_attack, average="weighted")},
    "severity": {"accuracy": accuracy_score(true_sev, pred_sev),
                 "macro_f1": f1_score(true_sev, pred_sev, average="macro")},
}
with open(MODELS + "unsw_multitask_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))